# Mass Balance Notebook Learning Guide

Notebook explained: `Mass_Balance_Equation.ipynb`

This guide explains the notebook in a study-note style: what each code cell is doing, how the algorithm flows, what the plots mean, and how to explain the analysis to a supervisor.

Important note: your attached reference PDF could not be read because the file at `C:\Users\owner\Downloads\Present_Data _Analysis Flows.pdf` is currently 0 bytes. I used the same kind of learning-guide structure you described: algorithm first, then cell-by-cell explanations.

---

## 1. Big Picture

The notebook studies the mass balance equation for confined HT1080 cell monolayers:

```text
d rho / dt + d(rho v_x) / dx = (k_prolif - k_loss) rho
```

In plain language:

- `rho(x,t)` is the local cell density.
- `v_x(x,t)` is the local horizontal velocity from optical flow.
- `d rho / dt` measures how density changes with time.
- `d(rho v_x) / dx` measures how transport changes density locally.
- `k_prolif(x,t)` is the local proliferation rate.
- `k_loss(x,t)` is loss/death/removal rate. In this notebook version, it is set to zero first.
- `residual` is the mismatch between the measured left side and the modeled right side.

The notebook is local in `x` and `time`, but not truly local in `y`. The data are binned across the full image height, so each value is an x-bin average or x-bin total over y.

Good supervisor sentence:

```
The mass-balance fields are calculated on an x-by-time grid. Each x-bin includes the full image height, so the analysis is local along x and time, but not y-resolved.
```

---

## 2. Algorithm Flow

The notebook follows this algorithm:

1. Import Python libraries.
2. Define file paths, physical constants, bin sizes, smoothing settings, and plotting settings.
3. Define helper functions for smoothing, binning, derivatives, plotting, and safe fitting.
4. Load Cellpose cell data, first-appearance/proliferation events, total counts, and optical-flow velocities.
5. Create the time axes:
   - `frames` for raw image frames.
   - `pair0s` for frame pairs used by optical flow.
6. Create the spatial x-bin grid.
7. Bin cell counts into `(frame, x-bin)`.
8. Convert counts into density `rho(x,t)`.
9. Bin optical-flow velocity into `v_x(x,t)`.
10. Bin first-appearance events into `(frame pair, x-bin)`.
11. Convert event counts into local proliferation rate `k_prolif(x,t)`.
12. Smooth density, velocity, and proliferation.
13. Calculate mass-balance terms:
    - `d rho / dt`
    - `rho v_x`
    - `d(rho v_x) / dx`
    - `LHS`
    - `RHS`
    - `residual`
    - effective loss estimate
14. Make plots to visualize the mass-balance fields.
15. Save outputs as CSV files.
16. Run local correlation analysis between source/divergence-like terms and `k_prolif`.

---

## 3. Key Variables

| Variable | Meaning | Shape / grid |
|---|---|---|
| `df_cells` | Cellpose cell table | one row per detected cell |
| `df_events` | first-appearance/proliferation events | one row per event |
| `df_counts` | total cell count per frame | one row per frame |
| `df_vraw` | optical-flow velocity table | long table by frame pair and x position |
| `frames` | frame numbers, usually 0 to 100 | 1D |
| `pair0s` | starting frame for frame pairs, usually 0 to 99 | 1D |
| `frame_times_h` | time for each frame in hours | 1D |
| `pair_times_h` | midpoint time for frame pairs | 1D |
| `x_centers_mm_centered` | x-bin centers, centered around image midpoint | 1D |
| `counts_xt` | cell counts in each x-bin and frame | `(time, x)` |
| `rho_xt` | cell density in each x-bin and frame | `(time, x)` |
| `vx_xt` | x velocity in each frame pair and x-bin | `(time pair, x)` |
| `kprolif_xt` | proliferation rate in each frame pair and x-bin | `(time pair, x)` |
| `rho_xt_use` | smoothed density used for derivatives | `(time, x)` |
| `vx_use` | smoothed velocity | `(time pair, x)` |
| `kprolif_use` | smoothed proliferation rate | `(time pair, x)` |
| `drho_dt_xt` | time derivative of density | `(time pair, x)` |
| `flux_xt` | `rho_mid * vx` | `(time pair, x)` |
| `dflux_dx_xt` | spatial derivative of flux | `(time pair, x)` |
| `lhs_xt` | `drho_dt_xt + dflux_dx_xt` | `(time pair, x)` |
| `rhs_xt` | `(kprolif - kloss) * rho_mid` | `(time pair, x)` |
| `residual_xt` | `lhs_xt - rhs_xt` | `(time pair, x)` |
| `kloss_eff_xt` | inferred effective loss needed to close equation | `(time pair, x)` |

---

## 4. Cell 1: Imports

Overall purpose: load the Python packages needed for file paths, arrays, tables, plotting, and smoothing.

```python
from pathlib import Path
```

Imports `Path`, which makes file paths easier and cleaner to write.

```python
import numpy as np
```

Imports NumPy. The notebook uses it for arrays, math, masks, derivatives, and reshaping.

```python
import pandas as pd
```

Imports pandas. The notebook uses it for CSV files and DataFrames.

```python
import matplotlib.pyplot as plt
```

Imports Matplotlib plotting tools.

```python
from scipy.ndimage import uniform_filter1d
```

Imports a simple 1D smoothing filter. This is used to smooth data along time and x.

```python
plt.rcParams["figure.dpi"] = 140
```

Sets the notebook display resolution for figures.

```python
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
```

Removes the top and right borders from plots, making the figures cleaner.

---

## 5. Cell 3: File Paths and Settings

Overall purpose: define where the data are located and set the physical/numerical parameters used in the analysis.

### File path block

```python
BASE_DIR = Path(r"C:\Users\owner\NADA Project\w400\s161_1\Raw")
```

This is the base folder for the dataset.

```python
FP_EVENTS = Path(...)
FP_CELLS  = Path(...)
FP_COUNTS = Path(...)
FP_VRAW   = Path(...)
```

These are the exact CSV files loaded later:

- `FP_EVENTS`: first-appearance events, used as proliferation events.
- `FP_CELLS`: Cellpose cell locations by frame.
- `FP_COUNTS`: total cell count per frame.
- `FP_VRAW`: optical-flow velocity profiles.

### Physical settings

```python
UM_PER_PX = 0.74
MM_PER_PX = UM_PER_PX / 1000.0
```

Each pixel is 0.74 micrometers. The code also converts that to millimeters.

```python
DT_H = 0.25
```

The time step is 0.25 hours, which is 15 minutes.

```python
X_CENTER_PX = None
```

The x-center is not given manually. The notebook later estimates it from the velocity table.

```python
IMAGE_H_PX = None
```

The image height is not given manually. The notebook later estimates it from cell centroid y-values.

### Grid settings

```python
DX_PX = 25
```

The image is divided into x-bins of 25 pixels.

```python
PROLIF_WINDOW_PAIRS = 5
```

The proliferation rate is estimated using a centered time window of 5 frame pairs. Since each pair is 0.25 h, this is 1.25 h.

### Smoothing settings

```python
SMOOTH_RHO_T = 3
SMOOTH_RHO_X = 3
SMOOTH_VX_T = 3
SMOOTH_VX_X = 3
SMOOTH_K_T = 1
SMOOTH_K_X = 3
```

These control how much smoothing is applied:

- density is smoothed in time and x,
- velocity is smoothed in time and x,
- proliferation is smoothed only in x.

Why smoothing matters:

```text
Derivatives amplify noise. Smoothing before calculating derivatives makes the mass-balance terms more stable.
```

### Loss term

```python
USE_ZERO_KLOSS = True
```

This tells the notebook to first assume `k_loss = 0`.

This means:

```text
RHS = k_prolif * rho
```

Then the notebook uses the residual to estimate an effective loss term.

### Numerical safety

```python
EPS = 1e-12
RHO_FLOOR = 1e-12
```

These tiny numbers prevent division by zero.

### Plotting settings

```python
SELECTED_PAIR_IDXS = [10, 30, 50, 70, 90]
```

These are selected time points used for line-profile plots.

```python
CMAP_MAIN = "viridis"
CMAP_SIGNED = "coolwarm"
```

`viridis` is used for positive fields. `coolwarm` is used for signed fields with positive and negative values.

```python
SAVE_FIGS = True
FIG_DPI = 300
```

Figures will be saved at 300 dpi.

```python
FIG_DIR = BASE_DIR / "mass_balance_plots"
FIG_DIR.mkdir(parents=True, exist_ok=True)
```

Creates the folder where figures are saved.

---

## 6. Cell 5: Helper Functions

Overall purpose: define reusable small functions so the later analysis cells stay readable.

### `smooth_xt`

```python
def smooth_xt(arr, size_t=1, size_x=1):
```

Defines a function that smooths a 2D array across time and x.

```python
out = np.asarray(arr, dtype=float).copy()
```

Converts the input into a float NumPy array and copies it.

```python
if size_t > 1:
    out = uniform_filter1d(out, size=size_t, axis=0, mode="nearest")
```

If time smoothing is requested, smooth along axis 0, which is the time axis.

```python
if size_x > 1:
    out = uniform_filter1d(out, size=size_x, axis=1, mode="nearest")
```

If x smoothing is requested, smooth along axis 1, which is the x axis.

```python
return out
```

Returns the smoothed array.

### `rolling_sum_centered_2d`

Overall: computes a centered rolling sum along time for each x-bin.

Important lines:

```python
kernel = np.ones(int(win), dtype=float)
```

Creates a window of ones. This is used for summing.

```python
for j in range(arr.shape[1]):
    out[:, j] = np.convolve(arr[:, j], kernel, mode="same")
```

Loops through each x-bin and applies the rolling sum over time.

Why this is used:

```text
Proliferation events are sparse, so the notebook counts events over a small time window instead of only one frame pair.
```

### `make_x_edges_px`

Overall: creates x-bin edges in pixels.

```python
edges = np.arange(0, width_px + dx_px, dx_px, dtype=float)
```

Creates bins from 0 to image width using the chosen bin size.

```python
if edges[-1] > width_px:
    edges[-1] = width_px
```

Makes sure the final bin edge does not go past the image boundary.

### `centers_from_edges`

Overall: converts bin edges into bin centers.

```python
return 0.5 * (edges[:-1] + edges[1:])
```

Each center is the average of the left and right bin edge.

### `bin_counts_by_frame`

Overall: counts how many cells or events fall into each x-bin for each frame.

```python
out = np.zeros((len(frame_values), len(x_edges_px) - 1), dtype=float)
```

Creates an empty `(time, x-bin)` table.

```python
frame_to_idx = {int(f): i for i, f in enumerate(frame_values)}
```

Creates a dictionary so frame numbers can be quickly converted into row indexes.

```python
for fr, grp in df.groupby(frame_col):
```

Loops through each frame.

```python
hist, _ = np.histogram(grp[x_col_px].values, bins=x_edges_px)
```

Counts how many x-values fall into each x-bin.

```python
out[frame_to_idx[fr], :] = hist
```

Stores that frame's histogram into the output array.

### `build_velocity_xt`

Overall: converts the long velocity table into a clean `(time pair, x-bin)` velocity array.

```python
out = np.full((len(pair0_values), len(x_edges_px) - 1), np.nan, dtype=float)
```

Creates an output array filled with `NaN`, because some bins may not have velocity values.

```python
for fr, grp in vraw.groupby("frame0"):
```

Loops through each optical-flow frame pair.

```python
bin_idx = np.digitize(xvals, x_edges_px) - 1
```

Assigns each velocity measurement to an x-bin.

```python
row[bi] = np.nanmean(m)
```

Calculates the average velocity in that bin, ignoring NaNs.

### `diff_x_centered`

Overall: calculates spatial derivative along x.

```python
out[:, 1:-1] = (arr[:, 2:] - arr[:, :-2]) / (2.0 * dx)
```

Uses centered differences for interior x-bins.

```python
out[:, 0] = (arr[:, 1] - arr[:, 0]) / dx
out[:, -1] = (arr[:, -1] - arr[:, -2]) / dx
```

Uses one-sided differences at the edges because there is no bin on one side.

### `robust_limits` and `robust_signed_limits`

Overall: choose better color limits for heatmaps.

These functions ignore extreme outliers by using percentiles.

### `plot_xt_map`

Overall: makes a heatmap where x is horizontal, time is vertical, and color shows the value.

Important plot line:

```python
im = ax.imshow(data, aspect="auto", origin="lower", extent=[...])
```

Displays the 2D array as an image.

```python
origin="lower"
```

Makes early time appear at the bottom and later time at the top.

```python
extent=[x_start, x_end, t_start, t_end]
```

Labels the heatmap axes in real units, not array indexes.

```python
plt.colorbar(im, ax=ax, pad=0.02)
```

Adds a colorbar so the color values can be interpreted.

### `safe_polyfit`

Overall: safely fits a straight line.

```python
m = np.isfinite(x) & np.isfinite(y)
```

Keeps only valid x-y pairs.

```python
if m.sum() < 3:
    return np.nan, np.nan
```

Avoids fitting a line if there are too few points.

```python
slope, intercept = np.polyfit(x[m], y[m], 1)
```

Fits a straight line:

```text
y = slope * x + intercept
```

---

## 7. Cell 7: Load Data and Build Axes

Overall purpose: load CSV files and create the frame/time/x coordinate systems.

```python
df_events = pd.read_csv(FP_EVENTS)
df_cells  = pd.read_csv(FP_CELLS)
df_counts = pd.read_csv(FP_COUNTS)
df_vraw   = pd.read_csv(FP_VRAW)
```

Reads the four main CSV files.

```python
print("events:", df_events.shape, df_events.columns.tolist())
```

Prints table size and column names. This is a sanity check.

### Frame axes

```python
frames = np.arange(int(df_counts["frame"].min()), int(df_counts["frame"].max()) + 1, dtype=int)
```

Creates all frame numbers from the first to last frame.

```python
pair0s = np.sort(df_vraw["frame0"].unique().astype(int))
```

Gets all starting frames used for optical-flow frame pairs.

```python
assert np.array_equal(frames, np.arange(0, 101))
assert np.array_equal(pair0s, np.arange(0, 100))
```

Checks that the notebook is working with the expected frame ranges.

```python
frame_times_h = frames * DT_H
```

Converts frame number into time in hours.

```python
pair_times_h = pair0s * DT_H + 0.5 * DT_H
```

Calculates midpoint time for each frame pair.

### X center

```python
if X_CENTER_PX is None:
    x_min_px = float(df_vraw["x_px"].min())
    x_max_px = float(df_vraw["x_px"].max())
    X_CENTER_PX = 0.5 * (x_min_px + x_max_px)
```

If the center was not manually set, estimate it as the midpoint of the x-range.

### Image geometry

```python
IMAGE_W_PX = int(np.floor(df_vraw["x_px"].max())) + 1
```

Estimates image width from the velocity table.

```python
IMAGE_H_PX = int(np.ceil(df_cells["centroid_y"].max())) + 1
```

Estimates image height from the highest cell y-coordinate.

### Add x coordinates

```python
df_cells["x_px"] = df_cells["centroid_x"].astype(float)
```

Copies cell x-centroid into a simpler column name.

```python
df_cells["x_px_centered"] = df_cells["x_px"] - float(X_CENTER_PX)
```

Centers x so that zero is near the middle of the field.

```python
df_cells["x_mm"] = df_cells["x_px"] * MM_PER_PX
df_cells["x_mm_centered"] = df_cells["x_px_centered"] * MM_PER_PX
```

Converts x from pixels to millimeters.

The same thing is done for `df_events`, using `first_x`.

### Common x bins

```python
x_edges_px = make_x_edges_px(width_px=IMAGE_W_PX, dx_px=DX_PX)
```

Creates x-bin edges.

```python
x_centers_px = centers_from_edges(x_edges_px)
```

Creates x-bin centers.

```python
x_centers_mm_centered = x_centers_px_centered * MM_PER_PX
```

Converts centered bin positions to millimeters.

### Bin areas

```python
bin_widths_mm = bin_widths_px * MM_PER_PX
image_h_mm = IMAGE_H_PX * MM_PER_PX
bin_areas_mm2 = bin_widths_mm * image_h_mm
```

Calculates each x-bin area in square millimeters.

This is important because density is:

```text
rho = cell count / bin area
```

---

## 8. Cell 9: Counts, Density, Velocity, and Proliferation

Overall purpose: create the main biological fields on the x-time grid.

### Total cell count

```python
total_count_t = df_counts.sort_values("frame")["n_cells"].values.astype(float)
```

Creates a 1D array of total cell counts over time.

### Counts per x-bin

```python
counts_xt = bin_counts_by_frame(...)
```

Counts how many cells are in each x-bin for each frame.

Shape:

```text
counts_xt = (number of frames, number of x-bins)
```

### Density

```python
rho_xt = counts_xt / np.maximum(bin_areas_mm2[None, :], EPS)
```

Converts count into density by dividing by bin area.

`np.maximum(..., EPS)` avoids division by zero.

### Midpoint density

```python
rho_mid_xt = 0.5 * (rho_xt[:-1, :] + rho_xt[1:, :])
```

Calculates density at the midpoint between two frames.

This is needed because velocity is defined between frames.

### Velocity

```python
vx_xt = build_velocity_xt(df_vraw, pair0_values=pair0s, x_edges_px=x_edges_px, vel_col="vx")
vy_xt = build_velocity_xt(df_vraw, pair0_values=pair0s, x_edges_px=x_edges_px, vel_col="vy")
```

Bins optical-flow velocities into the same x-grid.

```python
vx_xt = vx_xt / 1000.0
vy_xt = vy_xt / 1000.0
```

Converts velocity from micrometers per hour to millimeters per hour.

Only `vx_xt` is used in the 1D mass-balance equation.

### Event counts

```python
event_counts_xt = bin_counts_by_frame(...)
```

Counts first-appearance events in each x-bin and frame pair.

These events are used as a proxy for proliferation.

### Proliferation rate

```python
events_win_xt = rolling_sum_centered_2d(event_counts_xt, PROLIF_WINDOW_PAIRS)
```

Counts events over a small centered time window.

```python
count_mid_xt = 0.5 * (counts_xt[:-1, :] + counts_xt[1:, :])
```

Estimates local cell count at the midpoint of each frame pair.

```python
cell_hours_xt = count_mid_xt * DT_H
```

Converts cell counts into cell-hours.

```python
cell_hours_win_xt = rolling_sum_centered_2d(cell_hours_xt, PROLIF_WINDOW_PAIRS)
```

Sums cell-hours over the same time window used for events.

```python
kprolif_xt = events_win_xt / np.maximum(cell_hours_win_xt, EPS)
```

Calculates proliferation rate:

```text
k_prolif = number of division-like events / local cell-hours
```

Units:

```text
1 / hour
```

---

## 9. Cell 11: Mass Balance Calculations

Overall purpose: calculate the terms in the mass-balance equation.

### Smoothing

```python
rho_xt_use = smooth_xt(rho_xt, size_t=SMOOTH_RHO_T, size_x=SMOOTH_RHO_X)
```

Smooths density before derivative calculations.

```python
rho_mid_use = 0.5 * (rho_xt_use[:-1, :] + rho_xt_use[1:, :])
```

Calculates smoothed midpoint density.

```python
vx_use = smooth_xt(vx_xt, size_t=SMOOTH_VX_T, size_x=SMOOTH_VX_X)
```

Smooths x velocity.

```python
kprolif_use = smooth_xt(kprolif_xt, size_t=SMOOTH_K_T, size_x=SMOOTH_K_X)
```

Smooths proliferation rate.

### Time derivative

```python
drho_dt_xt = (rho_xt_use[1:, :] - rho_xt_use[:-1, :]) / DT_H
```

Calculates:

```text
d rho / dt
```

This tells how density changes between frames.

### Flux

```python
flux_xt = rho_mid_use * vx_use
```

Calculates density times velocity:

```text
flux = rho * v_x
```

This represents movement of cell density through space.

### Flux derivative

```python
dflux_dx_xt = diff_x_centered(flux_xt, dx=dx_mm_nominal)
```

Calculates:

```text
d(rho v_x) / dx
```

This is the local divergence of the flux.

Scientific meaning:

- If this term is positive or negative, it means transport is locally changing density.
- The exact compression/expansion interpretation depends on sign convention.

### Loss term

```python
if USE_ZERO_KLOSS:
    kloss_xt = np.zeros_like(kprolif_use)
```

Sets local loss to zero.

### Balance terms

```python
lhs_xt = drho_dt_xt + dflux_dx_xt
```

Calculates the left side:

```text
LHS = d rho / dt + d(rho v_x) / dx
```

```python
rhs_xt = (kprolif_use - kloss_xt) * rho_mid_use
```

Calculates the source side from proliferation and loss.

Because `k_loss = 0`, this is:

```text
RHS = k_prolif * rho
```

```python
residual_xt = lhs_xt - rhs_xt
```

Calculates the mismatch:

```text
residual = measured density change and transport - modeled proliferation source
```

### Effective loss

```python
kloss_eff_xt = np.where(
    rho_mid_use > RHO_FLOOR,
    -residual_xt / np.maximum(rho_mid_use, EPS),
    np.nan
)
```

This estimates the effective loss term that would be needed to close the equation.

If residual is not small, the notebook asks:

```text
What loss-like term would explain this mismatch?
```

### Closure metrics

```python
mask = np.isfinite(lhs_xt) & np.isfinite(rhs_xt)
```

Keeps only valid LHS/RHS pairs.

```python
lhs_vals = lhs_xt[mask]
rhs_vals = rhs_xt[mask]
```

Flattens valid values into two matching 1D arrays.

```python
corr = np.corrcoef(rhs_vals, lhs_vals)[0, 1]
```

Calculates correlation between LHS and RHS.

```python
slope, intercept = safe_polyfit(rhs_vals, lhs_vals)
```

Fits a line:

```text
LHS = slope * RHS + intercept
```

```python
rmse = np.sqrt(np.nanmean((lhs_vals - rhs_vals) ** 2))
```

Calculates average mismatch size.

---

## 10. Cell 13 Plot: Total Cell Count vs Time

Overall purpose: show whether total cell number increases over the experiment.

Important lines:

```python
fig, ax = plt.subplots(figsize=(6.8, 4.2))
```

Creates a plot.

```python
ax.plot(frame_times_h, total_count_t, lw=2.4)
```

Plots total segmented cells against time in hours.

```python
ax.set_xlabel("Time (h)")
ax.set_ylabel("Total segmented cells")
```

Labels the axes.

```python
ax.set_title("Total cell count vs time")
```

Gives the plot a title.

```python
fig.savefig(...)
```

Saves the figure if `SAVE_FIGS` is true.

How to explain this plot:

```text
This plot checks the global growth trend of the monolayer. If the curve rises, the total segmented cell count increases over time. This is a global check before looking at local mass-balance fields.
```

---

## 11. Cell 15 Plot: Heatmap Panel

Overall purpose: show the major mass-balance fields over x and time.

```python
fig, axs = plt.subplots(4, 2, figsize=(13, 16), constrained_layout=True)
```

Creates an 8-panel figure: 4 rows and 2 columns.

Each panel is a heatmap:

- x-axis: centered x position in mm.
- y-axis: time in hours.
- color: value of the field.

### Panel 1: `rho(x,t)`

Shows cell density over x and time.

Supervisor explanation:

```text
This panel shows where cells are denser or less dense along the confined monolayer over time.
```

### Panel 2: `v_x(x,t)`

Shows horizontal velocity from optical flow.

Supervisor explanation:

```text
This panel shows local motion along x. Positive and negative colors show motion direction.
```

### Panel 3: `k_prolif(x,t)`

Shows local proliferation rate.

Supervisor explanation:

```text
This panel shows where first-appearance events imply higher or lower local proliferation rate.
```

### Panel 4: `d rho / dt`

Shows local density accumulation or depletion over time.

Supervisor explanation:

```text
This panel tells where density is increasing or decreasing locally.
```

### Panel 5: `d(rho v_x) / dx`

Shows local flux divergence.

Supervisor explanation:

```text
This panel shows where transport contributes to local density gain or loss.
```

### Panel 6: `RHS = (k_prolif - k_loss) rho`

Shows the proliferation/loss source term. In this notebook version, `k_loss = 0`.

Supervisor explanation:

```text
This is the source term predicted from measured proliferation and local density.
```

### Panel 7: `Residual = LHS - RHS`

Shows mismatch between measured mass-balance left side and proliferation-based right side.

Supervisor explanation:

```text
This panel shows where proliferation alone does not explain the observed density dynamics and transport.
```

### Panel 8: `k_loss_eff(x,t)`

Shows inferred effective loss required to close the equation.

Supervisor explanation:

```text
This is not directly measured loss. It is the loss-like term needed mathematically to make the equation balance.
```

---

## 12. Cell 17 Plot: LHS vs RHS Scatter

Overall purpose: test whether the two sides of the mass-balance equation agree.

```python
ax.scatter(rhs_vals, lhs_vals, s=8, alpha=0.25)
```

Each point is one local `(time, x-bin)` value.

x-axis:

```text
RHS = (k_prolif - k_loss) rho
```

y-axis:

```text
LHS = d rho / dt + d(rho v_x) / dx
```

```python
ax.plot([mn - pad, mx + pad], [mn - pad, mx + pad], "--")
```

Draws the identity line.

Meaning:

```text
If a point lies on this line, LHS equals RHS exactly.
```

```python
ax.plot(xx, slope * xx + intercept, ...)
```

Draws the fitted trend line.

How to explain this plot:

```text
This plot checks mass-balance closure. Good agreement means local LHS values are close to local RHS values. The identity line is perfect agreement. The fitted line shows the average relationship between RHS and LHS.
```

What to say if points are scattered:

```text
Scatter indicates that proliferation alone, with k_loss set to zero, does not fully explain local density change and transport.
```

---

## 13. Cell 19 Plot: Selected Time Profiles

Overall purpose: show line profiles across x at selected time points.

```python
selected = [i for i in SELECTED_PAIR_IDXS if 0 <= i < len(pair_times_h)]
```

Keeps only selected time indices that are valid.

```python
labels = [f"{pair_times_h[i]:.2f} h" for i in selected]
```

Creates labels like `2.50 h`, `7.50 h`, etc.

```python
fig, axs = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
```

Creates six line-profile panels.

### Panel 1: selected-time `rho(x)`

Shows cell density across x at selected times.

Use this to explain where the monolayer is dense or sparse.

### Panel 2: selected-time `v_x(x)`

Shows velocity across x at selected times.

Use this to explain how flow varies spatially.

### Panel 3: selected-time `k_prolif(x)`

Shows proliferation rate across x at selected times.

Use this to explain whether proliferation is higher in certain regions.

### Panel 4: selected-time `d rho / dt`

Shows local density change across x.

Use this to explain where density is increasing or decreasing.

### Panel 5: selected-time `d(rho v_x) / dx`

Shows local flux divergence across x.

Use this to explain transport-driven density effects.

### Panel 6: LHS/RHS/residual at one representative time

Compares:

- LHS,
- RHS,
- residual.

Supervisor explanation:

```text
This panel lets us see at one time point where the equation matches well and where it does not.
```

---

## 14. Cell 21 Plot: Time-Averaged Effective Loss Profile

Overall purpose: summarize `k_loss_eff` over time as a function of x.

```python
valid = np.isfinite(kloss_eff_xt)
```

Finds valid values.

```python
n_valid = np.sum(valid, axis=0)
```

Counts valid time points for each x-bin.

```python
kloss_eff_mean_x = np.nanmean(kloss_eff_xt, axis=0)
```

Calculates the mean effective loss at each x-bin.

```python
kloss_eff_sem_x = kloss_eff_std_x / np.sqrt(np.maximum(n_valid, 1))
```

Calculates standard error of the mean.

```python
ax.plot(x_centers_mm_centered, kloss_eff_mean_x, ...)
```

Plots the mean profile.

```python
ax.fill_between(..., alpha=0.25)
```

Adds a shaded SEM band.

How to explain this plot:

```text
This plot shows the average inferred loss-like term across x. It is not directly measured death or loss; it is the effective term needed to explain the residual mismatch in the mass-balance equation.
```

Important caution:

```text
This is time-averaged across the experiment, so it is not a local x-time scatter analysis.
```

---

## 15. Cell 23: Save Wide Outputs

Overall purpose: save the main calculated arrays as CSV files.

```python
OUT_DIR = BASE_DIR / "mass_balance_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
```

Creates the output folder.

```python
pd.DataFrame({...}).to_csv(OUT_DIR / "x_bins.csv", index=False)
```

Saves x-bin positions and bin areas.

```python
pd.DataFrame({"frame": frames, "t_h": frame_times_h}).to_csv(...)
```

Saves frame time information.

```python
def save_wide(mat, row_ids, x_centers_mm, path, row_name="row"):
```

Defines a helper function to save 2D arrays in wide format.

In wide format:

- each row is a time point,
- each column is an x position.

```python
cols = [f"{x:.3f}" for x in x_centers_mm]
```

Uses x-position values as column names.

```python
df.insert(0, row_name, row_ids)
```

Adds the frame or frame-pair number as the first column.

Then the notebook saves:

- `rho_xt.csv`
- `vx_xt.csv`
- `kprolif_xt.csv`
- `drho_dt_xt.csv`
- `dflux_dx_xt.csv`
- `rhs_xt.csv`
- `residual_xt.csv`
- `kloss_eff_xt.csv`

Supervisor explanation:

```text
This cell exports the main fields so they can be inspected outside the notebook or reused in later analysis.
```

---

## 16. Cell 25: Mass Balance Summary Report

Overall purpose: create a readable text summary of the analysis and save it as `mass_balance_summary.txt`.

This cell is long, but it is built from repeated blocks.

### Helper `_stats`

```python
def _stats(arr):
```

Defines a small function to summarize an array.

It calculates:

- number of valid values,
- mean,
- standard deviation,
- min,
- max,
- median.

```python
arr = arr[np.isfinite(arr)]
```

Removes invalid values before calculating statistics.

### Helper `_fmt_stats`

```python
def _fmt_stats(name, arr, unit=""):
```

Formats the stats into readable text.

This is why the summary report looks organized.

### Helper `_region_mask`

```python
def _region_mask(x_mm, region):
```

Creates masks for center, edge, and intermediate regions.

Important caution:

```text
In this cell, the region thresholds use 50 and 100 while the variable name is x_mm. If x is truly in mm, the thresholds should probably be 0.050 and 0.100. This looks like an old micrometer-threshold convention. Check this before using region summaries seriously.
```

### Required variables

```python
required_vars = [...]
missing = [v for v in required_vars if v not in globals()]
```

Checks that earlier notebook cells were run.

### Metadata recovery

The cell recovers `X_CENTER_PX`, `IMAGE_W_PX`, and `IMAGE_H_PX` if needed.

This makes the summary more robust.

### `lines = []`

Creates an empty list of text lines.

The cell keeps appending text to this list, then joins it into one report.

### Header and equation section

This section writes:

- report title,
- equation,
- what each field represents,
- that `k_loss = 0` in this version.

### Basic data/grid info

Reports:

- number of frames,
- time interval,
- pixel size,
- x center,
- image width/height,
- x-bin width,
- number of x-bins,
- proliferation window.

### Total cell count section

Reports initial and final cell counts and percent increase.

### Global variable summary

Reports statistics for:

- density,
- velocity,
- flux,
- derivatives,
- proliferation,
- LHS,
- RHS,
- residual,
- effective loss.

### Region summaries

Reports the same statistics for center, intermediate, and edge regions.

Again, check the threshold units carefully.

### Selected time summary

Reports values for one selected frame pair.

This helps explain one time slice in detail.

### Closure metrics

Reports:

- correlation between LHS and RHS,
- fitted slope and intercept,
- RMSE,
- mean LHS,
- mean RHS,
- mean residual,
- mean inferred loss at center/edges.

### Interpretation block

Adds human-readable interpretation.

```python
summary_text = "\n".join(lines)
```

Combines all text lines into one long report.

```python
with open(SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(summary_text)
```

Saves the summary text file.

---

## 17. Cell 27: Long-Format Export Table

Overall purpose: create one table where each row is one local `(time pair, x-bin)` point.

This is very useful for later correlation analysis.

```python
MB_DIR = BASE_DIR / "mass_balance_equation"
MB_DIR.mkdir(parents=True, exist_ok=True)
```

Creates output folder.

```python
frame0_col = pair0s
frame1_col = pair0s + 1
```

Stores the two frames in each frame pair.

```python
n_t, n_x = lhs_xt.shape
```

Gets number of time pairs and x-bins.

```python
frame0_long = np.repeat(frame0_col, n_x)
```

Repeats each frame number for every x-bin.

```python
x_center_mm_long = np.tile(x_centers_mm_centered, n_t)
```

Repeats the full x-axis for every time point.

This is how the notebook creates matched rows.

### Counts and densities

```python
counts_t0 = counts_xt[:-1, :]
counts_t1 = counts_xt[1:, :]
counts_mid = 0.5 * (counts_t0 + counts_t1)
```

Stores counts at frame 0, frame 1, and midpoint.

```python
rho_t0 = rho_xt_use[:-1, :]
rho_t1 = rho_xt_use[1:, :]
```

Stores density at the two frames.

### Main DataFrame

```python
df_mb_values = pd.DataFrame({...})
```

Creates a table with coordinates and all important variables.

Each row contains:

- frame pair,
- time,
- x position,
- cell counts,
- density,
- velocity,
- flux,
- derivative terms,
- proliferation,
- loss,
- LHS,
- RHS,
- residual,
- effective loss.

### Region label

```python
df_mb_values["region"] = np.where(...)
```

Adds center/edge/intermediate labels.

Important caution:

```text
This region labeling also uses 50 and 100 while the column is x_center_mm. If x is in mm, check whether these should be 0.050 and 0.100.
```

### Save full table

```python
df_mb_values.to_csv(mb_csv, index=False)
```

Saves the table.

### Summary table

```python
df_mb_summary = pd.DataFrame([...])
```

Creates a compact summary of valid counts and ranges for each variable.

### One selected time slice

```python
PAIR_TO_VIEW = 50
df_mb_one_time = df_mb_values[df_mb_values["frame0"] == PAIR_TO_VIEW].copy()
```

Extracts one time point for closer inspection.

---

## 18. Cell 29: Quick `Vx` vs `k_prolif` Correlation

Overall purpose: test whether local velocity is related to local proliferation.

This is not the source/divergence analysis yet. It is a first-pass velocity-proliferation check.

### Import scipy if available

```python
try:
    from scipy.stats import pearsonr, spearmanr
    use_scipy_stats = True
except Exception:
    use_scipy_stats = False
```

Uses scipy for correlations and p-values if available.

### Check variables

```python
if "vx_use" not in globals() or "kprolif_use" not in globals():
    raise RuntimeError(...)
```

Stops if the mass-balance cells were not run first.

### Choose x-axis units

The code uses `x_centers_mm_centered` if available.

### Output folder

```python
out_dir = BASE_DIR / "mass_balance_equation" / "vx_kprolif_correlation"
```

Creates a separate folder for this analysis.

### Match arrays

```python
vx_arr = np.asarray(vx_use, dtype=float)
k_arr = np.asarray(kprolif_use, dtype=float)
```

Converts velocity and proliferation to numeric arrays.

```python
if vx_arr.shape != k_arr.shape:
    raise RuntimeError(...)
```

Checks both arrays are on the same grid.

### Create matched table

```python
df_corr = pd.DataFrame({...})
```

Creates one row per local `(time pair, x-bin)` point.

Columns include:

- `frame0`,
- `t_mid_h`,
- x position,
- `vx`,
- `abs_vx`,
- `k_prolif`.

### Clean values

```python
df_corr = df_corr.replace([np.inf, -np.inf], np.nan).dropna(...)
```

Removes invalid points.

### Correlation helper

```python
def get_corr(x, y, method="pearson"):
```

Calculates Pearson or Spearman correlation.

Pearson:

```text
tests linear relationship
```

Spearman:

```text
tests monotonic/rank relationship
```

### Correlations

The code calculates:

- `vx` vs `k_prolif`,
- `abs(vx)` vs `k_prolif`.

### Scatter plot 1: `Vx` vs `k_prolif`

Each point is one local x-time pair.

The line is a simple linear fit.

How to explain:

```text
This plot asks whether regions moving faster in the x direction tend to have higher or lower proliferation.
```

### Scatter plot 2: `|Vx|` vs `k_prolif`

This ignores direction and only uses speed magnitude.

How to explain:

```text
This plot asks whether stronger local motion, regardless of direction, is associated with proliferation.
```

---

## 19. Extension Cell 32: Prepare Local Source/Divergence vs `k_prolif` Pairs

Overall purpose: create the clean local paired dataset for source/divergence correlation.

```python
required = [...]
```

Lists variables that must exist from earlier cells.

```python
missing = [v for v in required if v not in globals()]
```

Checks which required variables are missing.

```python
if missing:
    raise RuntimeError(...)
```

Stops the cell if earlier mass-balance cells have not been run.

```python
TERM_TO_TEST = "dflux_dx_xt"
```

Chooses the source/divergence-like term to compare with proliferation.

Default choice:

```text
dflux_dx_xt = d(rho v_x)/dx
```

This is a good term for local flux divergence.

```python
term_options = {...}
```

Defines the possible terms:

- `dflux_dx_xt`: flux divergence,
- `lhs_xt`: full left side source needed by density change and transport,
- `residual_xt`: imbalance after subtracting proliferation source.

Important caution:

```text
Use residual carefully because residual already contains k_prolif indirectly through RHS. For a cleaner biological comparison, dflux_dx_xt is safer.
```

```python
source_arr, source_title, source_unit = term_options[TERM_TO_TEST]
```

Gets the selected array and labels.

```python
source_arr = np.asarray(source_arr, dtype=float)
k_arr = np.asarray(kprolif_use, dtype=float)
```

Converts the selected term and proliferation to numeric arrays.

```python
if source_arr.shape != k_arr.shape:
    raise RuntimeError(...)
```

Checks that both arrays are matched on the same x-time grid.

```python
n_t, n_x = source_arr.shape
```

Gets number of time points and x-bins.

```python
corr_dir = BASE_DIR / "mass_balance_equation" / "source_kprolif_local_correlation"
```

Creates a separate output folder for the new analysis.

### Create the paired table

```python
df_local_pairs = pd.DataFrame({...})
```

Each row is one matched local point.

```python
"frame0": np.repeat(pair0s, n_x)
```

Repeats each frame pair for all x-bins.

```python
"t_mid_h": np.repeat(pair_times_h, n_x)
```

Repeats each time value for all x-bins.

```python
"x_center_mm": np.tile(x_centers_mm_centered, n_t)
```

Repeats the x-axis for every time point.

```python
"source_or_divergence": source_arr.reshape(-1)
```

Flattens the selected source/divergence array into one column.

```python
"abs_source_or_divergence": np.abs(source_arr.reshape(-1))
```

Stores absolute magnitude, ignoring sign.

```python
"k_prolif_per_h": k_arr.reshape(-1)
```

Stores matched proliferation values.

### Clean invalid values

```python
df_local_pairs = (
    df_local_pairs
    .replace([np.inf, -np.inf], np.nan)
    .dropna(...)
    .reset_index(drop=True)
)
```

Turns infinity into `NaN`, removes bad rows, and resets row numbering.

### Save

```python
df_local_pairs.to_csv(pairs_csv, index=False)
```

Saves the local paired table.

How to explain:

```text
This cell creates the actual dataset used for local correlation. It does not average over x or time. Each row is one matching x-bin and time point.
```

---

## 20. Extension Cell 34: Calculate Local Correlations

Overall purpose: calculate Pearson and Spearman correlations between source/divergence and proliferation.

```python
try:
    from scipy.stats import pearsonr, spearmanr
```

Tries to load scipy statistical functions.

```python
have_scipy_stats = True
```

Means p-values can be calculated.

```python
except Exception:
    have_scipy_stats = False
```

If scipy is missing, the code still calculates correlations but p-values are not available.

### Correlation helper

```python
def simple_corr(x, y, method):
```

Defines a helper for Pearson or Spearman correlation.

```python
x = np.asarray(x, dtype=float)
y = np.asarray(y, dtype=float)
```

Converts inputs to numeric arrays.

```python
ok = np.isfinite(x) & np.isfinite(y)
```

Keeps only valid pairs.

```python
x, y = x[ok], y[ok]
```

Applies the valid mask.

```python
if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
    return np.nan, np.nan
```

Avoids meaningless correlations.

```python
if have_scipy_stats:
```

Uses scipy if possible.

```python
pearsonr(x, y)
```

Calculates Pearson correlation and p-value.

Pearson meaning:

```text
Does y change linearly with x?
```

```python
spearmanr(x, y)
```

Calculates Spearman correlation and p-value.

Spearman meaning:

```text
As x ranks higher, does y also tend to rank higher or lower?
```

### Actual correlations

```python
pearson_source, p_pearson_source = simple_corr(...)
```

Pearson correlation between signed source/divergence and proliferation.

```python
spearman_source, p_spearman_source = simple_corr(...)
```

Spearman correlation between signed source/divergence and proliferation.

```python
pearson_abs, p_pearson_abs = simple_corr(...)
```

Pearson correlation between absolute source/divergence magnitude and proliferation.

```python
spearman_abs, p_spearman_abs = simple_corr(...)
```

Spearman correlation between absolute magnitude and proliferation.

### Summary table

```python
df_corr_summary = pd.DataFrame([...])
```

Creates a readable table containing:

- comparison name,
- method,
- correlation value,
- p-value,
- number of valid points.

```python
df_corr_summary.to_csv(summary_csv, index=False)
```

Saves the summary.

How to explain:

```text
This cell gives the numerical strength of association between local source/divergence and local proliferation. Pearson checks linear association, while Spearman checks rank-based monotonic association.
```

---

## 21. Extension Cell 36 Plot: Local Source/Divergence vs `k_prolif`

Overall purpose: visually show the paired local correlation results.

```python
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
```

Creates two side-by-side scatter plots.

### Left plot: signed source/divergence vs `k_prolif`

```python
axs[0].scatter(df_local_pairs["source_or_divergence"], df_local_pairs["k_prolif_per_h"], s=10, alpha=0.22)
```

Each dot is one local matched `(time, x-bin)` point.

x-axis:

```text
source_or_divergence
```

y-axis:

```text
k_prolif
```

```python
s=10
```

Small dots, useful because there are many points.

```python
alpha=0.22
```

Makes points transparent so dense areas appear darker.

```python
axs[0].set_xlabel(...)
axs[0].set_ylabel(...)
```

Labels axes.

```python
axs[0].set_title(...)
```

Adds Pearson and Spearman values to the title.

How to explain the left plot:

```text
This plot tests whether the signed local source/divergence term is associated with local proliferation. If the point cloud slopes upward, higher source/divergence values tend to have higher proliferation. If it slopes downward, they tend to have lower proliferation. If it is very scattered, there is no simple relationship.
```

### Right plot: absolute source/divergence vs `k_prolif`

```python
axs[1].scatter(df_local_pairs["abs_source_or_divergence"], df_local_pairs["k_prolif_per_h"], s=10, alpha=0.22)
```

Uses absolute value of source/divergence.

This ignores direction/sign and only asks about strength.

How to explain the right plot:

```text
This plot tests whether the magnitude of local compression/expansion or imbalance is related to proliferation, regardless of sign.
```

### Save figure

```python
scatter_png = corr_dir / f"{TERM_TO_TEST}_kprolif_scatter_plots.png"
fig.savefig(scatter_png, dpi=300, bbox_inches="tight")
```

Saves the figure.

### Optional fit line explanation

If you add a linear fit line with:

```python
m, b = np.polyfit(df_local_pairs["source_or_divergence"], df_local_pairs["k_prolif_per_h"], 1)
```

then `m` is the slope and `b` is the intercept.

The line represents:

```text
k_prolif = m * source_or_divergence + b
```

Supervisor explanation:

```text
The fitted line is only a visual guide for the average linear trend. The correlation values are the quantitative summary.
```

---

## 22. Extension Cell 38 Plot: Optional Binned Trend

Overall purpose: simplify the scatter plot by binning source/divergence values and plotting average proliferation in each bin.

```python
N_BINS = 8
```

Uses 8 bins.

```python
if df_local_pairs["source_or_divergence"].nunique() < 3:
```

Checks whether there are enough different values to bin.

```python
df_bin = df_local_pairs.copy()
```

Makes a copy so the original table is not changed.

```python
df_bin["source_bin"] = pd.qcut(..., q=N_BINS, duplicates="drop")
```

Splits source/divergence values into bins with roughly equal numbers of points.

```python
df_binned = df_bin.groupby("source_bin", observed=True).agg(...)
```

Calculates one summary row per bin.

For each bin it calculates:

- mean source/divergence,
- mean `k_prolif`,
- standard deviation of `k_prolif`,
- number of points.

```python
df_binned["sem_k_prolif"] = ...
```

Calculates standard error of the mean:

```text
SEM = standard deviation / sqrt(number of points)
```

```python
ax.errorbar(...)
```

Plots mean proliferation per bin with error bars.

How to explain this plot:

```text
The binned plot summarizes the dense scatter plot. Each point is a group of local values with similar source/divergence. The y-value is the mean proliferation rate in that bin, and the error bar shows uncertainty in the mean.
```

Important caution:

```text
This plot is easier to read than the scatter, but binning choices can affect the apparent trend.
```

---

## 23. How to Explain the Whole Notebook to a Supervisor

Short version:

```text
I use Cellpose detections to calculate local density along x and optical flow to estimate local velocity along x. Then I calculate the mass-balance terms d rho/dt and d(rho v_x)/dx. I compare the measured left side of the equation with the proliferation-based right side. I also calculate the residual and an inferred effective loss term to see where proliferation alone does not explain density dynamics.
```

For the local source/proliferation extension:

```text
In the extension, I test whether local source/divergence patterns are associated with local proliferation. I pair each source/divergence value with the k_prolif value at the same x-bin and time point. This avoids reducing the analysis to one average profile across x. However, the current notebook fields are already averaged over y, so this is local in x and time, not full x-y local analysis.
```

---

## 24. What Each Plot Proves and Does Not Prove

### Total cell count plot

Shows global cell number change over time.

Does not show local spatial effects.

### Heatmap panel

Shows where density, velocity, proliferation, transport, residual, and inferred loss vary over x and time.

Does not prove causation.

### LHS vs RHS scatter

Tests mass-balance closure.

If points follow identity, LHS and RHS match well.

If points do not follow identity, proliferation with `k_loss = 0` is not enough to explain the measured dynamics.

### Selected time profile plot

Shows spatial structure at selected times.

Useful for explaining specific time snapshots.

### Effective loss profile

Shows time-averaged inferred loss across x.

It is not directly measured cell death.

### `Vx` vs `k_prolif` scatter

Tests whether local velocity or speed is associated with proliferation.

It does not test flux divergence directly.

### Source/divergence vs `k_prolif` scatter

Tests whether local source/divergence-like values are associated with local proliferation.

This is the main new local correlation analysis.

### Binned trend plot

Summarizes dense scatter points into a simpler trend.

Good for presentation, but affected by bin choices.

---

## 25. Main Limitations to Say Clearly

- Correlation does not prove causation.
- The current mass-balance grid is local in x and time, not truly x-y local.
- Divergence/source terms can be noisy because derivatives amplify noise.
- `k_prolif` depends on the quality of first-appearance or division detection.
- Optical flow measures image-feature motion, not direct single-cell tracking.
- Source/divergence and `k_prolif` must be on the same spatial and temporal grid.
- Sign convention for divergence must be checked carefully.
- Smoothing choices can change derivative values.
- Binning choices can change apparent trends.
- Edge effects can influence density, velocity, divergence, and proliferation.
- The region labels in the current notebook should be checked for mm vs micrometer threshold consistency.

---

## 26. A Simple Class Explanation

```text
First, the notebook loads cell positions, proliferation-like events, total cell counts, and optical-flow velocity. Then it bins everything along x and time. From cell counts it calculates density rho. From optical flow it calculates v_x. From first-appearance events divided by local cell-hours it calculates k_prolif.

After that, the notebook calculates the mass-balance equation terms. It computes d rho/dt, the flux rho v_x, and the flux derivative d(rho v_x)/dx. These form the left side. The right side is k_prolif times rho because k_loss is set to zero in this version. The difference between the two sides is the residual.

Finally, the extension asks whether local source/divergence values are associated with local proliferation. It makes a matched table where every row is one time point and one x-bin, then calculates Pearson and Spearman correlations and makes scatter plots.
```

